In [28]:
keyToNep = {
  "a": "\u093E",# ा
  "b": "\u092C",# ब
  "c": "\u091B",# छ
  "d": "\u0926",# द
  "e": "\u0947",# े
  "f": "\u0909",# उ
  "g": "\u0917",# ग
  "h": "\u0939",# ह
  "i": "\u093F",# ि
  "j": "\u091C",# ज
  "k": "\u0915",# क
  "l": "\u0932",# ल
  "m": "\u092E",# म
  "n": "\u0928",# न
  "o": "\u094B",# ो
  "p": "\u092A",# प
  "q": "\u091F",# ट
  "r": "\u0930",# र
  "s": "\u0938",# स
  "t": "\u0924",# त
  "u": "\u0941",# ु
  "v": "\u0935",# व
  "w": "\u094C",# ौ
  "x": "\u0921",# ड
  "y": "\u092F",# य
  "z": "\u0937",# ष
  "A": "\u0906",# आ
  "B": "\u092D",# भ
  "C": "\u091A",# च
  "D": "\u0927",# ध
  "E": "\u0948",# ै
  "F": "\u090A",# ऊ
  "G": "\u0918",# घ
  "H": "\u0905",# अ
  "I": "\u0940",# ी
  "J": "\u091D",# झ
  "K": "\u0916",# ख
  "L": "\u0933",# ळ
  "M": "\u0902",# ं
  "N": "\u0923",# ण
  "O": "\u0913",# ओ
  "P": "\u092B",# फ
  "Q": "\u0920",# ठ
  "R": "\u0943",# ृ
  "S": "\u0936",# श
  "T": "\u0925",# थ
  "U": "\u0942",# ू
  "V": "\u0901",# ँ
  "W": "\u0914",# औ
  "X": "\u0922",# ढ
  "Y": "\u091E",# ञ
  "Z": "\u090B",# ऋ
  "0": "\u0966",# ०
  "1": "\u0967",# १
  "2": "\u0968",# २
  "3": "\u0969",# ३
  "4": "\u096A",# ४
  "5": "\u096B",# ५
  "6": "\u096C",# ६
  "7": "\u096D",# ७
  "8": "\u096E",# ८
  "9": "\u096F",# ९
  "^": "\u005E",# ^
  "`": "\u093D",# ऽ
  "~": "\u093C",# ़
  "_": "\u0952",# ॒
  "+": "\u200C",# 
  "=": "\u200D",#
  "[": "\u0907",# इ
  "{": "\u0908",# ई
  "]": "\u090F",# ए
  "}": "\u0910", # ऐ
  "\\": "\u0950",#/ ॐ
  "|": "\u0903",# ः
  "<": "\u0919",# ङ
  ".": "\u0964",# ।
  ">": "\u0965",# ॥
  "/": "\u094D",# ्
  "?": "\u003F",# ?
}

def formatKey(key):
  return keyToNep[key]

def roman2nepali(roman: str):
  return "".join([formatKey(char) for char in roman])


In [29]:
import pandas as pd
import numpy as np

letters = [
    "ब", "छ", "द", "उ", "ग", "ह", "ज", "क", "ल", "म", "न", "प", 
    "ट", "र", "स", "त", "व", "ड", "य", "ष", "आ", "भ", "च", "ध", "ऊ", 
    "घ", "अ", "झ", "ख", "ळ", "ण", "ओ", "फ", "ठ", "श", "थ", 
    "औ", "ढ", "ञ", "ऋ", "इ", "ई", "ए", "ऐ", "ङ", "ऑ"
]

extensions = [
    "ा", "ि", "ी", "ु", "ू", "ृ", "े", "ै", "ो", "ौ",  # vowel signs
    "ं", "ँ", "ः",  # anusvara, chandrabindu, visarga
    "़", "ऽ", "्", "॒"  # nukta, avagraha, halant, anudatta
]

symbols = [
    "^", "`", "~", "_", "+", "=", "[", "{", "]", "}", "\\", "|", "<", ".", ">", "/", "?", "!", "'", '"', "।", ",","‘","“"
]

all = pd.read_csv("words.csv")
common = pd.read_csv("common_words_in_poem.csv")
all.shape

(113393, 1)

In [30]:
all['words'].head()

0        अ
1       अ:
2       अँ
3    अँएरो
4    अँकरा
Name: words, dtype: object

In [31]:
# -*- coding: utf-8 -*-
"""
Complete rhyme detection for Devanagari (Hindi/Sanskrit)
with fixed syllable segmentation and all advanced features.
"""

import re

# ========== FIXED SYLLABLE SEGMENTATION ==========
def get_syllables(word):
    """
    Parse Devanagari word into syllables.
    Rules:
    - Independent vowel → single syllable.
    - Consonant + optional halant + optional vowel sign → one syllable.
    - A consonant that is NOT followed by halant and is followed by another consonant
      indicates that the first consonant has its inherent vowel, so it ends a syllable.
    """
    vowels = set('अआइईउऊऋएऐओऔ')
    vowel_signs = set('ािीुूृेैोौ')
    halant = '्'
    
    syllables = []
    current = []
    i = 0
    n = len(word)
    
    while i < n:
        ch = word[i]
        if ch in vowels:                     # Independent vowel
            if current:
                syllables.append(''.join(current))
                current = []
            current.append(ch)
            syllables.append(''.join(current))
            current = []
            i += 1
        elif ch == halant:                   # Halant – stays in current syllable
            if current:
                current.append(ch)
            i += 1
        elif ch in vowel_signs:              # Vowel sign – ends current syllable
            if current:
                current.append(ch)
                syllables.append(''.join(current))
                current = []
            else:
                # Should not happen, but fallback
                syllables.append(ch)
            i += 1
        else:                               # Consonant
            # If current syllable exists and it does NOT end with a halant,
            # then the previous consonant had an inherent vowel → finalize it.
            if current and current[-1] != halant:
                syllables.append(''.join(current))
                current = []
            current.append(ch)
            i += 1
    
    if current:
        syllables.append(''.join(current))
    return syllables

# ========== VOWEL MAPPING & CONFIGURABLE MODES ==========
VOWEL_MAP = {
    'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ii', 'उ': 'u', 'ऊ': 'uu',
    'ऋ': 'ri', 'ए': 'e', 'ऐ': 'ai', 'ओ': 'o', 'औ': 'au',
    'ा': 'aa', 'ि': 'i', 'ी': 'ii', 'ु': 'u', 'ू': 'uu',
    'ृ': 'ri', 'े': 'e', 'ै': 'ai', 'ो': 'o', 'ौ': 'au'
}

def get_vowel_family(raw_vowel, mode):
    if mode == 'strict':
        return raw_vowel
    families = {}
    if mode == 'loose':
        families = {'i': ['i','ii'], 'u': ['u','uu']}
    elif mode == 'very_loose':
        families = {
            'a': ['a','aa'],
            'i': ['i','ii'],
            'u': ['u','uu'],
            'e': ['e','ai'],
            'o': ['o','au']
        }
    else:
        families = {'i': ['i','ii'], 'u': ['u','uu']}
    for fam, members in families.items():
        if raw_vowel in members:
            return fam
    return raw_vowel

def get_vowel_sound(syl, mode, ignore_nasal):
    if ignore_nasal:
        syl = syl.replace('ं', '').replace('ँ', '')
    if syl in VOWEL_MAP and len(syl) == 1:
        raw = VOWEL_MAP[syl]
    else:
        raw = 'a'
        for matra, sound in VOWEL_MAP.items():
            if matra in syl and len(matra) == 1:
                raw = sound
                break
    return get_vowel_family(raw, mode)

def get_consonant_pattern(syl, ignore_nasal):
    if ignore_nasal:
        syl = syl.replace('ं', '').replace('ँ', '')
    for v in VOWEL_MAP:
        syl = syl.replace(v, '')
    return syl

# ========== CONSONANT SIMILARITY ==========
CONSONANT_GROUPS = {
    'velar': set('कखगघङ'),
    'palatal': set('चछजझञ'),
    'retroflex': set('टठडढण'),
    'dental': set('तथदधन'),
    'labial': set('पफबभम'),
    'semivowel': set('यरलव'),
    'fricative': set('शषसह')
}

def consonant_similarity(c1, c2):
    if c1 == c2:
        return 1.0
    if len(c1) == 1 and len(c2) == 1:
        for group in CONSONANT_GROUPS.values():
            if c1 in group and c2 in group:
                return 0.5
        return 0.0
    common = sum(1 for a, b in zip(c1, c2) if a == b)
    if len(c1) == len(c2):
        return (common / len(c1)) * 0.8
    return 0.2 if common > 0 else 0.0

def consonant_bonus(cons1, cons2):
    if cons1 == cons2:
        return 0.5
    sonorants = set('यरलवह')
    if (cons1 == '' and len(cons2) == 1 and cons2 in sonorants) or \
       (cons2 == '' and len(cons1) == 1 and cons1 in sonorants):
        return 0.25
    return consonant_similarity(cons1, cons2) * 0.5

# ========== SYLLABLE WEIGHT ==========
def is_heavy_syllable(syl, vowel_sound):
    long_vowels = {'aa', 'ii', 'uu', 'ri', 'e', 'ai', 'o', 'au'}
    if vowel_sound in long_vowels:
        return True
    if '्' in syl:
        return True
    return False

# ========== WEIGHTED MIDDLE MATCH ==========
def weighted_middle_match(middle_pairs, mode, ignore_nasal):
    if not middle_pairs:
        return 0.0
    total_weight = 0.0
    match_weight = 0.0
    for i, (s1, s2) in enumerate(middle_pairs):
        weight = 0.8 ** i
        total_weight += weight
        v1 = get_vowel_sound(s1, mode, ignore_nasal)
        v2 = get_vowel_sound(s2, mode, ignore_nasal)
        if v1 == v2:
            match_weight += weight
    return (match_weight / total_weight) * 2.0 if total_weight else 0.0

# ========== MAIN RHYME SCORE ==========
def rhyme_score(w1, w2, mode='loose', ignore_nasal=True, strict_alignment=None):
    """
    Compute rhyme score (0 to ~8). Higher = better rhyme.
    If strict_alignment is True, both first and last aligned vowels must match.
    If False, allow last-match-only (as before).
    By default, strict_alignment = (mode != 'very_loose').
    """
    w1s = get_syllables(w1)
    w2s = get_syllables(w2)
    if not w1s or not w2s:
        return 0.0

    # Set strict_alignment default based on mode
    if strict_alignment is None:
        strict_alignment = (mode != 'very_loose')

    # Syllable difference penalty (light, kept for safety)
    syl_diff = abs(len(w1s) - len(w2s))
    syl_penalty = 0.0
    if syl_diff >= 2:
        syl_penalty = -0.5   # mild penalty
    elif syl_diff == 1:
        syl_penalty = -0.2

    min_len = min(len(w1s), len(w2s))
    aligned = [(w1s[-(i+1)], w2s[-(i+1)]) for i in range(min_len)]
    if not aligned:
        return 0.0

    last_pair = aligned[0]
    first_pair = aligned[-1]
    middle_pairs = aligned[1:-1]

    last_v1 = get_vowel_sound(last_pair[0], mode, ignore_nasal)
    last_v2 = get_vowel_sound(last_pair[1], mode, ignore_nasal)
    first_v1 = get_vowel_sound(first_pair[0], mode, ignore_nasal)
    first_v2 = get_vowel_sound(first_pair[1], mode, ignore_nasal)
    last_match = (last_v1 == last_v2)
    first_match = (first_v1 == first_v2)

    if first_match and last_match:
        base_score = 3.0
        middle_bonus = weighted_middle_match(middle_pairs, mode, ignore_nasal)
    elif not first_match and last_match:
        if strict_alignment:
            # For loose/strict mode, this is not allowed → score 0
            return 0.0
        else:
            base_score = 1.5
            middle_bonus = weighted_middle_match(middle_pairs, mode, ignore_nasal)
    else:
        base_score = -2.0
        middle_bonus = 0.0

    score = base_score + middle_bonus + syl_penalty

    # Consonant bonus (only if strict_alignment or last_match)
    if not strict_alignment or (first_match and last_match):
        cons1 = get_consonant_pattern(last_pair[0], ignore_nasal)
        cons2 = get_consonant_pattern(last_pair[1], ignore_nasal)
        score += consonant_bonus(cons1, cons2)

    # Syllable weight bonus
    heavy1 = is_heavy_syllable(last_pair[0], last_v1)
    heavy2 = is_heavy_syllable(last_pair[1], last_v2)
    if heavy1 == heavy2:
        score += 0.3

    # Length bonus (very small)
    if score >= 1.0:
        len_diff = abs(len(w1) - len(w2))
        if len_diff == 0:
            score += 0.5
        elif len_diff == 1:
            score += 0.25
        elif len_diff == 2:
            score += 0.1

    return max(0.0, score)

def rhymes(w1, w2, mode='loose', ignore_nasal=True):
    """Return True if rhyme_score meets threshold (2.5)."""
    return rhyme_score(w1, w2, mode, ignore_nasal) >= 2.5

# ========== TEST ==========
if __name__ == '__main__':
    test_pairs = [
        ("राम", "धाम"),
        ("जल", "फल"),
        ("रोपए", "खोपए"),
        ("प्रेम", "रेम"),
        ("पानी", "सानी"),
        ("रामायण", "राम"),
        ("रोपए", "खोए"),
        ("राम", "रामी"),
        ("गीत", "प्रीत"),
        ("कमला", "अमल"),
        ("एक", "मेक"),       # should rhyme
        ("का", "बा"),
        ("सुख", "सूख"),
        ("कि", "की"),
        ("के", "कै"),
        ("को", "कौ"),
    ]

    print("=== RHYME DETECTION TEST (mode='loose', ignore_nasal=True) ===")
    for w1, w2 in test_pairs:
        score = rhyme_score(w1, w2)
        result = rhymes(w1, w2)
        print(f"'{w1}' vs '{w2}': {result} (score = {score:.2f})")

=== RHYME DETECTION TEST (mode='loose', ignore_nasal=True) ===
'राम' vs 'धाम': True (score = 4.30)
'जल' vs 'फल': True (score = 4.30)
'रोपए' vs 'खोपए': True (score = 6.30)
'प्रेम' vs 'रेम': True (score = 3.90)
'पानी' vs 'सानी': True (score = 4.30)
'रामायण' vs 'राम': False (score = 0.00)
'रोपए' vs 'खोए': False (score = 0.00)
'राम' vs 'रामी': False (score = 0.00)
'गीत' vs 'प्रीत': True (score = 3.90)
'कमला' vs 'अमल': False (score = 0.00)
'एक' vs 'मेक': True (score = 4.05)
'का' vs 'बा': True (score = 3.80)
'सुख' vs 'सूख': True (score = 4.30)
'कि' vs 'की': True (score = 4.30)
'के' vs 'कै': False (score = 0.00)
'को' vs 'कौ': False (score = 0.00)


In [32]:
common_words_in_poems = []
excluding = list("123456789")
with open("lspd.txt", "r") as f:
    corpus = f.read()
    corpuslist = corpus.split()
    for token in corpuslist:
        if token.strip() in symbols+excluding:
            continue
        else:
            token = token.strip()
            word = ""
            for c in token:
                if c in symbols+excluding:
                    continue
                else:
                    word+=c

            common_words_in_poems.append(word)
common_words_in_poems = pd.Series(common_words_in_poems).value_counts().index.to_numpy()

In [33]:
for w in common_words_in_poems:
    print(w)

यो
पनी
छ
भनी
म
त
र
पनि
कन
हो
गरी
सब्
तहाँ
भयो
न
ता
जसै
छन्
भो
गया
ति
खुप्
गर्या
हे
त्यो
तसै
ती
भै
यहाँ
आज
भया
जो
हुन्
सब
भनि
एक्
गरि
क्या
इ
बिन्ति
दिया
अब
गयो
बिन्ती
यस्तो
जब
मेरो
फेर्
तहिं
सित
एक
हुकुम्
महाँ
देखि
भन्या
के
तिमी
राम्
सबै
थिया
भई
विर्
मन्
कहाँ
ए
नै
वचन्
गर्न
रघुनाथ्
हवस्
बहुत्
तिम्रो
सरी
गै
सीता
थियो
सुनि
गई
लौ
अति
हुन्छ
बात्
खुसि
तिमि
पो
थ्यैं
यस्ता
हूकुम्
छैन
ताहाँ
जहाँ
मेरा
अघि
रे
विषे
सिता
रावण्
ऐल्हे
मनमा
भनेर
तिमिले
आयो
कि
यै
किन
हनुमान्
झैं
आया
लाग्यो
भाइ
परी
खुस्
यति
मलाई
हुन
धरी
जल्दि
हूँ
काम्
अगाडी
मेरी
तेस्
येती
अनेक्
साम्ने
बक्सनु
सुनी
विस्तार्
नाथ्
दिन्
गर्नु
लिया
मन
आत्मा
ली
खूसी
सकल
सो
अरु
कसरी
पर्या
लक्ष्मण्
हजुरमा
फेरि
होला
अहिले
लिई
ताहिं
जति
भन्छन्
यस्
गर्यो
देख्या
कुरा
पर्यो
भइ
लाग्या
को
पछि
ठुलो
आमा
तँ
वनमा
ई
सुन्दर
आँखा
राम्का
समेत्
क्यै
प्रभु
घर
हरी
ठूलो
जल्दी
पुत्र
जाऊ
साह्रै
कुन्
तर
गरेर
दुःख
ताप्
श्री
संसार
राज्य
हात्
दिन
झन्
मैले
उ
आई
सहजै
प्रभुजिले
उपर्
मुना
येति
ताहीं
राज्
ऋषिले
मनले
हुँदा
जुन्
केही
भन्न्या
बाली
जान
अवश्य
शोक्
फौज्
हेर
मात्र


In [34]:
print(get_syllables("रामायण"))

['रा', 'मा', 'य', 'ण']


In [35]:
word_syllables = []
for word in common_words_in_poems:
    word_syllables.append({word: get_syllables(word)})

In [40]:
w = np.random.choice(all.words)
print("Word:",w)
rhyming_words = []
for wd in common.words:
    score = rhyme_score(w,wd,mode="loose",strict_alignment=True)
    if score>3.5:
        rhyming_words.append({wd: score })
rhyming_words.sort(key=lambda x: list(x.values())[0], reverse=True)
[list(d.keys())[0] for d in rhyming_words]

Word: पकाउनु


['गराउनु',
 'बनाउनु',
 'बताउनु',
 'पठाउनु',
 'मराउनू',
 'बनाउनू',
 'जनाउनु',
 'बचाउनु',
 'खटाउनु',
 'खटाउनू',
 'लगाउनु',
 'अह्राउनु',
 'सह्राउनु',
 'लाउनु',
 'पाउनू',
 'आउनु',
 'आऊनू',
 'आऊनु',
 'खसालनू',
 'भाउज्यू',
 'महापशु',
 'जराहरु',
 'महामरु',
 'रस-मुटु',
 'रह्याँछू',
 'बच्चाहरु',
 'डकार्दछु',
 'प्रजाहरु',
 'रह्याँछु',
 'लोकपाल्हरु',
 'राखनू',
 'राखनु',
 'पशुहरु',
 'चरुहरु',
 'तरुहरु',
 'रहेंछु',
 'नलुकाऊ',
 'जनमन–आँशु',
 'ब्रह्महत्याहरू',
 'जनहरू',
 'रत्नहरु',
 'जनहरु',
 'हँसाऊ',
 'नकराऊ',
 'नसताऊ',
 'अन्नहरु',
 'रङ्गहरु',
 'संझाउ',
 'गणहरू',
 'खरहरू']

In [37]:
vowel_group_tests = [
    # Short vs long 'i' – now rhymes
    ("कि", "की", True),
    ("मिल", "मील", True),
    # Short vs long 'u' – now rhymes
    ("गु", "गू", True),
    ("सुख", "सूख", True),   # Note: सुख vs सूख – consonants match (स + ख), vowel family matches (u)
    # But 'े' vs 'ै' still do NOT rhyme (different families)
    ("के", "कै", False),
    # 'ो' vs 'ौ' still do NOT rhyme
    ("को", "कौ", False),
    ("क", "का", False),
]

print("\n=== VOWEL GROUPING TESTS (LOOSE_VOWEL_GROUPING = True) ===")
for w1, w2, expected in vowel_group_tests:
    score = rhyme_score(w1, w2)
    result = rhymes(w1, w2)
    status = "✅" if result == expected else "❌"
    print(f"{status} '{w1}' vs '{w2}': expected {expected}, got {result} (score {score:.2f})")


=== VOWEL GROUPING TESTS (LOOSE_VOWEL_GROUPING = True) ===
✅ 'कि' vs 'की': expected True, got True (score 4.30)
✅ 'मिल' vs 'मील': expected True, got True (score 4.30)
✅ 'गु' vs 'गू': expected True, got True (score 4.30)
✅ 'सुख' vs 'सूख': expected True, got True (score 4.30)
✅ 'के' vs 'कै': expected False, got False (score 0.00)
✅ 'को' vs 'कौ': expected False, got False (score 0.00)
✅ 'क' vs 'का': expected False, got False (score 0.00)
